# District-wise Participation Heatmap
## Maharashtra Mental Health Sensitisation Programme — DMER × iCALL, TISS

**Author:** Amruta Gaikwad  
**Affiliation:** iCALL, Tata Institute of Social Sciences (TISS), Mumbai  
**Project:** DMER Maharashtra × iCALL Mental Health Sensitisation Programme (2023–2025)

---

## Background

The DMER Maharashtra × iCALL, TISS Mental Health Sensitisation Programme was a state-level initiative delivered across 24 government medical colleges in Maharashtra, aimed at improving mental health literacy and reducing stigma among medical students. Data were collected as part of a pre-post evaluation study involving 3,000+ participants.

This notebook produces an **interactive choropleth map** visualising the district-wise distribution of student participation in the sensitisation sessions. The visualisation serves two purposes:

1. **Programme monitoring** — identifying districts with lower participation to guide follow-up outreach and resource allocation
2. **Policy communication** — presenting geographic reach to state-level stakeholders (including DMER) in an accessible, non-technical format

Districts not covered by the programme are rendered in grey to clearly distinguish programme reach from geographic gaps.

## Workflow

| Step | Description |
|------|-------------|
| 1 | Install and import libraries |
| 2 | Load Maharashtra district boundary GeoJSON |
| 3 | Load programme participation data (CSV) |
| 4 | Clean and align district names across both datasets |
| 5 | Merge geospatial and participation data |
| 6 | Prepare colour encoding for missing vs present data |
| 7 | Build and style interactive choropleth map |
| 8 | Export as standalone HTML and static PNG |

## Step 1 — Install libraries

| Library | Purpose |
|---|---|
| `pandas` | Data manipulation and CSV loading |
| `geopandas` | Loading and handling GeoJSON boundary data |
| `plotly` | Interactive choropleth visualisation |
| `mapclassify` | Choropleth classification schemes (required by geopandas) |
| `kaleido` | Static image export (PNG) from Plotly figures |

In [ ]:
%pip install pandas geopandas plotly mapclassify kaleido

## Step 2 — Load geospatial boundary data

We load a GeoJSON file containing the administrative boundaries for all 36 districts of Maharashtra. `geopandas` reads this into a **GeoDataFrame** — a pandas DataFrame with an additional `geometry` column holding each district's polygon coordinates.

We inspect the district name column carefully here, as exact string matching will be required when merging with the participation data in Step 5.

In [ ]:
import geopandas as gpd
import pandas as pd

# Load Maharashtra district boundary GeoJSON
# Contains polygon geometries for all 36 districts
gdf = gpd.read_file('maharashtra.geojson')

print(f"Total districts in boundary file: {gdf.shape[0]}")
print(f"Columns: {list(gdf.columns)}\n")
gdf[['district', 'geometry']].head(10)

## Step 3 — Load programme participation data

The participation dataset is a CSV exported from programme administrative records. Each row represents one district with the following key columns:

- `District` — district name (used as the merge key)
- `N` — number of medical students who participated in sensitisation sessions
- `%` — percentage of enrolled students who participated

Only districts hosting one of the 24 government medical colleges in the programme sample have entries. The remaining districts will appear as `NaN` after merging and will be rendered in grey on the map.

> **Note on data availability:** The participation CSV contains programme administrative data and is not publicly available. See `data/data_sources.md` for details.

In [ ]:
# Load participation data
# Each row = one district; key columns are 'District', 'N' (count), '%' (proportion)
try:
    df_participation = pd.read_csv('data/Maharashtra_DMER_Student_Outreach.csv')
    print(f"Districts with participation data: {df_participation.shape[0]}")
    print(f"Columns: {list(df_participation.columns)}\n")
    display(df_participation.head(10))
except FileNotFoundError:
    print("Data file not found. See data/data_sources.md for instructions on obtaining the data.")

## Step 4 — Clean and align district names

Before merging, district names must be consistent across both datasets. Common issues include:
- Leading or trailing whitespace
- Inconsistent capitalisation  
- Historical name differences (e.g., `Osmanabad` vs `Dharashiv` — Maharashtra officially renamed several districts in 2023)

We standardise both name columns and explicitly check for mismatches so no districts are silently dropped during the merge.

In [ ]:
# Standardise district name columns: strip whitespace, consistent title case
gdf['district_clean'] = gdf['district'].str.strip().str.title()
df_participation['district_clean'] = df_participation['District'].str.strip().str.title()

# Check for name mismatches — participation districts not found in GeoJSON
geojson_districts = set(gdf['district_clean'])
participation_districts = set(df_participation['district_clean'])

unmatched = participation_districts - geojson_districts
if unmatched:
    print(f"Districts in participation data NOT matched in GeoJSON ({len(unmatched)}):")
    for d in sorted(unmatched):
        print(f"  - {d}")
    print("\nManual corrections will be applied in the next cell.")
else:
    print("All participation districts matched successfully in GeoJSON.")

print(f"\nTotal GeoJSON districts: {len(geojson_districts)}")
print(f"Districts with participation data: {len(participation_districts)}")
print(f"Districts without participation data (will appear grey): {len(geojson_districts - participation_districts)}")

In [ ]:
# Manual name corrections for known mismatches
# Osmanabad was officially renamed Dharashiv by Maharashtra government in 2023.
# Depending on the GeoJSON version, either name may appear — adjust direction accordingly.
name_corrections = {
    'Dharashiv': 'Osmanabad',  # update if your GeoJSON uses the new name
}

df_participation['district_clean'] = df_participation['district_clean'].replace(name_corrections)

# Confirm all mismatches resolved
remaining = set(df_participation['district_clean']) - set(gdf['district_clean'])
print(f"Unmatched districts after corrections: {len(remaining)}")
if remaining:
    print("Still unmatched — review these manually:", remaining)

## Step 5 — Merge geospatial and participation data

We perform a **left merge** keeping all 36 GeoJSON districts as the base. Districts without participation data receive `NaN` values for `N` and `%`.

A left merge is appropriate here because every district should appear on the map — including those not covered by the programme — to accurately represent geographic reach and gaps.

In [ ]:
# Left merge: all 36 GeoJSON districts retained; participation data joined where available
merged_gdf = gdf.merge(
    df_participation[['district_clean', 'N', '%']],
    on='district_clean',
    how='left'
)

# Ensure N is numeric (coerce any non-numeric entries to NaN)
merged_gdf['N'] = pd.to_numeric(merged_gdf['N'], errors='coerce')

# Summary statistics
n_with_data    = merged_gdf['N'].notna().sum()
n_without_data = merged_gdf['N'].isna().sum()
print(f"Districts with participation data : {n_with_data}")
print(f"Districts without data (grey)     : {n_without_data}")
print(f"Total students across all districts: {merged_gdf['N'].sum():.0f}")
print(f"Participation range: {merged_gdf['N'].min():.0f} – {merged_gdf['N'].max():.0f} students")

merged_gdf[['district_clean', 'N', '%']].sort_values('N', ascending=False).head(10)

## Step 6 — Prepare colour encoding

We use a custom colour scale to communicate two distinct states:

| Colour | Meaning |
|---|---|
| **Grey** | District has no programme coverage (no government medical college in sample) |
| **Orange → Deep red** | District has participation data; intensity scales with student count |

Plotly's continuous colour scale does not natively support a special missing-data colour, so we assign a sentinel value of `-999` to `NaN` districts and map it to grey at position `0` of the scale. The actual data range begins at a near-zero offset (`1e-9`).

In [ ]:
# Assign sentinel value to districts with no data
# -999 is outside the real data range and will be mapped to grey
merged_gdf['color_value'] = merged_gdf['N'].fillna(-999)

max_val = merged_gdf['N'].max()

# Custom colour scale
# Position 0.0   → grey         (sentinel -999: no programme coverage)
# Position ~0    → light orange (lowest actual participation)
# Position 1.0   → deep red     (highest participation)
color_scale = [
    [0,    'rgb(200, 200, 200)'],       # grey: no data
    [1e-9, 'rgba(255, 165, 0, 0.4)'],  # light orange: low participation
    [1,    'rgb(180, 0, 0)']            # deep red: high participation
]

print(f"Colour scale spans: -999 (no data) → {max_val:.0f} (max participation)")

## Step 7 — Build interactive choropleth map

We use `plotly.express.choropleth` with the GeoDataFrame's geometry column directly as the GeoJSON source. Key design decisions:

- **`fitbounds="locations"`** — auto-zooms to Maharashtra's extent, removing surrounding Asia context
- **`visible=False`** on `update_geos` — hides basemap/coastlines so only district polygons are shown
- **Hover tooltips** display district name, participant count, and percentage on mouse-over
- **Title and colour bar** labelled for non-technical stakeholder audiences

In [ ]:
import plotly.express as px

fig = px.choropleth(
    merged_gdf,
    geojson=merged_gdf.geometry,
    locations=merged_gdf.index,
    color='color_value',
    hover_name='district_clean',
    hover_data={
        'N': True,
        '%': True,
        'color_value': False       # hide sentinel column from tooltip
    },
    color_continuous_scale=color_scale,
    range_color=[-999, max_val],
    scope='asia',
    center={'lat': 19.7515, 'lon': 75.7139},
    labels={
        'N': 'Students participated',
        '%': 'Participation rate (%)'
    }
)

# Zoom to Maharashtra extent; hide basemap
fig.update_geos(fitbounds='locations', visible=False)

# Layout: title + colour bar labels + minimal margins
fig.update_layout(
    margin={'r': 0, 't': 60, 'l': 0, 'b': 0},
    title={
        'text': (
            'Medical Student Participation in Mental Health Sensitisation Sessions<br>'
            '<sub>DMER Maharashtra × iCALL, TISS | District-wise, 2023–2025</sub>'
        ),
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 14}
    },
    coloraxis_colorbar={
        'title': 'Students<br>participated',
        'tickvals': [0, max_val * 0.25, max_val * 0.5, max_val * 0.75, max_val],
        'ticktext': [
            '0',
            f'{max_val * 0.25:.0f}',
            f'{max_val * 0.5:.0f}',
            f'{max_val * 0.75:.0f}',
            f'{max_val:.0f}'
        ]
    }
)

fig.show()

## Step 8 — Export outputs

We export two versions:
1. **Interactive HTML** — standalone file that works in any browser without Python; suitable for sharing with non-technical stakeholders
2. **Static PNG** — for inclusion in reports, presentations, or policy documents

In [ ]:
import plotly.io as pio

# Interactive standalone HTML
# include_plotlyjs=True embeds Plotly library so file works fully offline
fig.write_html(
    'maharashtra_heatmap.html',
    include_plotlyjs=True,
    full_html=True
)
print("Interactive HTML exported → maharashtra_heatmap.html")

# Static PNG (requires kaleido)
pio.write_image(fig, 'maharashtra_heatmap.png', width=1000, height=800, scale=2)
print("Static PNG exported → maharashtra_heatmap.png")

## Summary

This notebook produced an interactive district-level choropleth map of Maharashtra showing the geographic distribution of medical student participation in the DMER × iCALL mental health sensitisation programme.

**Key observations:**
- Districts with government medical colleges show variation in participation, reflecting differences in college size, enrolment, and local implementation
- Districts shaded grey have no government medical college in the programme sample — representing geographic gaps potentially relevant to future scale-up planning
- The interactive HTML output allows stakeholders to hover over individual districts for exact counts and rates

**Methodological note:** Participation counts reflect students who attended sensitisation sessions across the evaluation period (2023–2025). Percentage figures are calculated against enrolled student totals provided by college administrators.

**Reproducibility:** All code is self-contained in this notebook. To reproduce the map, obtain the participation data (see `data/data_sources.md`), place the CSV and GeoJSON files as described, and run all cells in order.